In [1]:
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit

In [2]:
# Load the balanced dataset generated in the previous step
input_file = "../data/processed/yelp_reviews_sampled_50k.csv"
print(f"Loading dataset: {input_file} ...")

df = pd.read_csv(input_file)

# Ensure the text column is a string and handle potential missing values
df["text"] = df["text"].astype(str)

print(f"Loaded rows: {len(df)}")
print("Columns:", list(df.columns))

Loading dataset: ../data/processed/yelp_reviews_sampled_50k.csv ...
Loaded rows: 50000
Columns: ['review_id', 'business_id', 'stars', 'text']


In [3]:
# Filter out very short reviews (< 5 tokens)
# Here we use a fast whitespace split to estimate token counts
print("Computing token counts and filtering short reviews (< 5 tokens) ...")

df["token_count"] = df["text"].apply(lambda x: len(x.split()))
initial_count = len(df)

df_filtered = df[df["token_count"] >= 5].copy()
filtered_count = len(df_filtered)

print(f"Filtering completed. Removed {initial_count - filtered_count} short reviews.")
print(f"Remaining rows: {filtered_count}")

Computing token counts and filtering short reviews (< 5 tokens) ...
Filtering completed. Removed 4 short reviews.
Remaining rows: 49996


In [4]:
# Split by business_id to avoid leakage across splits
# Step 1: split into 80% train and 20% temp
print("Splitting dataset by business_id (80/10/10) ...")

gss1 = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, temp_idx = next(gss1.split(df_filtered, groups=df_filtered["business_id"]))

train_data = df_filtered.iloc[train_idx].copy()
temp_data = df_filtered.iloc[temp_idx].copy()

# Step 2: split the 20% temp into 10% val and 10% test
gss2 = GroupShuffleSplit(n_splits=1, test_size=0.5, random_state=42)
val_idx, test_idx = next(gss2.split(temp_data, groups=temp_data["business_id"]))

val_data = temp_data.iloc[val_idx].copy()
test_data = temp_data.iloc[test_idx].copy()

print("--- Split Summary ---")
print(f"Train rows: {len(train_data)}")
print(f"Val rows:   {len(val_data)}")
print(f"Test rows:  {len(test_data)}")

Splitting dataset by business_id (80/10/10) ...
--- Split Summary ---
Train rows: 39750
Val rows:   5025
Test rows:  5221


In [5]:
# Save the split datasets
print("Saving split datasets to disk ...")

train_path = "../data/processed/train_data.csv"
val_path = "../data/processed/val_data.csv"
test_path = "../data/processed/test_data.csv"

train_data.to_csv(train_path, index=False)
val_data.to_csv(val_path, index=False)
test_data.to_csv(test_path, index=False)

print("Saved:")
print(f"- {train_path}")
print(f"- {val_path}")
print(f"- {test_path}")

Saving split datasets to disk ...
Saved:
- ../data/processed/train_data.csv
- ../data/processed/val_data.csv
- ../data/processed/test_data.csv


In [6]:
# Optional sanity checks: confirm no business_id overlaps across splits
train_biz = set(train_data["business_id"].unique())
val_biz = set(val_data["business_id"].unique())
test_biz = set(test_data["business_id"].unique())

print("Business overlap checks:")
print("Train ∩ Val:", len(train_biz.intersection(val_biz)))
print("Train ∩ Test:", len(train_biz.intersection(test_biz)))
print("Val ∩ Test:", len(val_biz.intersection(test_biz)))

Business overlap checks:
Train ∩ Val: 0
Train ∩ Test: 0
Val ∩ Test: 0
